In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS olist_project.silver")

DataFrame[]

In [0]:
from pyspark.sql.functions import to_timestamp, col

orders_bronze = spark.table("olist_project.bronze.orders")

orders_silver = (orders_bronze
    .dropDuplicates(["order_id"])
    .filter(col("order_id").isNotNull())
    .withColumn("purchase_ts", to_timestamp("order_purchase_timestamp"))
    .withColumn("approved_ts", to_timestamp("order_approved_at"))
    .withColumn("delivered_carrier_ts", to_timestamp("order_delivered_carrier_date"))
    .withColumn("delivered_customer_ts", to_timestamp("order_delivered_customer_date"))
    .withColumn("estimated_delivery_ts", to_timestamp("order_estimated_delivery_date"))
    .select("order_id", "customer_id", "order_status",
            "purchase_ts", "approved_ts", "delivered_carrier_ts",
            "delivered_customer_ts", "estimated_delivery_ts")
)

orders_silver.write.format("delta").mode("overwrite") \
    .saveAsTable("olist_project.silver.orders")

print(f"Bronze rows: {orders_bronze.count()}, Silver rows: {orders_silver.count()}")

Bronze rows: 99441, Silver rows: 99441


In [0]:
customers_silver = (spark.table("olist_project.bronze.customers")
    .dropDuplicates(["customer_id"])
    .filter(col("customer_id").isNotNull())
    .withColumnRenamed("customer_zip_code_prefix", "zip_code_prefix")
    .withColumnRenamed("customer_city", "city")
    .withColumnRenamed("customer_state", "state")
    .select("customer_id", "customer_unique_id", "zip_code_prefix", "city", "state")
)
customers_silver.write.format("delta").mode("overwrite") \
    .saveAsTable("olist_project.silver.customers")

In [0]:
items_silver = (spark.table("olist_project.bronze.order_items")
    .filter(col("order_id").isNotNull() & col("product_id").isNotNull())
    .withColumnRenamed("price", "item_price")
)
items_silver.write.format("delta").mode("overwrite") \
    .saveAsTable("olist_project.silver.order_items")

In [0]:
payments_silver = (spark.table("olist_project.bronze.order_payments")
    .filter(col("order_id").isNotNull())
)
payments_silver.write.format("delta").mode("overwrite") \
    .saveAsTable("olist_project.silver.order_payments")

In [0]:
products_silver = (spark.table("olist_project.bronze.products")
    .dropDuplicates(["product_id"])
    .filter(col("product_id").isNotNull())
)
products_silver.write.format("delta").mode("overwrite") \
    .saveAsTable("olist_project.silver.products")

In [0]:
sellers_silver = (spark.table("olist_project.bronze.sellers")
    .dropDuplicates(["seller_id"])
    .filter(col("seller_id").isNotNull())
)
sellers_silver.write.format("delta").mode("overwrite") \
    .saveAsTable("olist_project.silver.sellers")